# Loan Approval Prediction — Baseline Decision Tree
**Goal:** Establish a clean baseline model and store all metrics for later comparison.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json, warnings

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
)

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
print("All imports loaded ✓")

In [ ]:
# ── 1. Load the dataset ──────────────────────────────────────────────────
df = pd.read_csv("loan_data.csv")
print(f"Shape: {df.shape}")
df.head()

In [ ]:
# ── 2. Exploratory Data Analysis ─────────────────────────────────────────
print("Data types:\n")
print(df.dtypes)
print("\n\nBasic statistics (numerical):\n")
df.describe()

In [ ]:
# ── 2b. Categorical column overview ──────────────────────────────────────
cat_cols = df.select_dtypes(include="object").columns.tolist()
print(f"Categorical columns: {cat_cols}\n")
for col in cat_cols:
    print(f"{col}: {df[col].unique()}")
    print(f"  value counts:\n{df[col].value_counts()}\n")

In [ ]:
# ── 2c. Distribution plots ───────────────────────────────────────────────
num_cols = df.select_dtypes(include=np.number).columns.tolist()

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.ravel()
for i, col in enumerate(num_cols):
    axes[i].hist(df[col].dropna(), bins=30, edgecolor="black", alpha=0.7)
    axes[i].set_title(col)
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
plt.suptitle("Distributions of Numerical Features", fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── 3. Missing values & class imbalance ──────────────────────────────────
print("Missing values per column:\n")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No missing values found ✓")

print(f"\n\nTotal missing cells: {missing.sum()}")
print(f"Total cells: {df.size}")
print(f"Missing %: {missing.sum() / df.size * 100:.2f}%")

print("\n\n── Class distribution (loan_status) ──")
counts = df["loan_status"].value_counts()
print(counts)
print(f"\nImbalance ratio (majority / minority): {counts.max() / counts.min():.2f}")

fig, ax = plt.subplots(figsize=(5, 4))
counts.plot.bar(ax=ax, color=["steelblue", "salmon"], edgecolor="black")
ax.set_title("Class Distribution — loan_status")
ax.set_xlabel("Class")
ax.set_ylabel("Count")
for i, v in enumerate(counts):
    ax.text(i, v + 200, str(v), ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── 4. Encode categorical variables ──────────────────────────────────────
label_encoders = {}
df_enc = df.copy()

for col in cat_cols:
    le = LabelEncoder()
    df_enc[col] = le.fit_transform(df_enc[col])
    label_encoders[col] = le
    print(f"{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

print("\nEncoded dataframe shape:", df_enc.shape)
df_enc.head()

In [ ]:
# ── 5. Stratified train / test split ─────────────────────────────────────
TARGET = "loan_status"
X = df_enc.drop(columns=[TARGET])
y = df_enc[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {X_train.shape[0]}  |  Test size: {X_test.shape[0]}")
print(f"Train class distribution:\n{y_train.value_counts(normalize=True).round(4)}")
print(f"\nTest class distribution:\n{y_test.value_counts(normalize=True).round(4)}")

In [ ]:
# ── 6. Decision Tree with Stratified K-Fold Cross Validation ─────────────
dt = DecisionTreeClassifier(random_state=42)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = ["f1", "precision", "recall", "accuracy"]
cv_results = cross_validate(dt, X_train, y_train, cv=skf, scoring=scoring,
                            return_train_score=False)

print("── Stratified 5-Fold CV Results ──\n")
for metric in scoring:
    scores = cv_results[f"test_{metric}"]
    print(f"{metric:>10s}:  mean = {scores.mean():.4f}  ±  {scores.std():.4f}   {scores.round(4)}")

In [ ]:
# ── 7. Evaluate on hold-out test set ─────────────────────────────────────
dt.fit(X_train, y_train)
y_pred = dt.predict(X_test)

test_acc  = accuracy_score(y_test, y_pred)
test_f1   = f1_score(y_test, y_pred)
test_prec = precision_score(y_test, y_pred)
test_rec  = recall_score(y_test, y_pred)

print("── Test-Set Evaluation ──\n")
print(f"  Accuracy : {test_acc:.4f}")
print(f"  F1-score : {test_f1:.4f}")
print(f"  Precision: {test_prec:.4f}")
print(f"  Recall   : {test_rec:.4f}")
print(f"\n{classification_report(y_test, y_pred, target_names=['Rejected (0)', 'Approved (1)'])}")

# Confusion matrix
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred,
                                        display_labels=["Rejected", "Approved"],
                                        cmap="Blues", ax=ax)
ax.set_title("Confusion Matrix — Decision Tree (Baseline)")
plt.tight_layout()
plt.show()

In [ ]:
# ── 8. Feature importance & tree depth ───────────────────────────────────
importances = pd.Series(dt.feature_importances_, index=X.columns).sort_values(ascending=False)

print(f"Tree depth: {dt.get_depth()}")
print(f"Number of leaves: {dt.get_n_leaves()}\n")
print("Feature importances:\n")
print(importances.round(4).to_string())

fig, ax = plt.subplots(figsize=(10, 5))
importances.plot.barh(ax=ax, color="steelblue", edgecolor="black")
ax.set_title("Feature Importance — Decision Tree (Baseline)")
ax.set_xlabel("Importance")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# ── 9. Save all baseline metrics to JSON ─────────────────────────────────
baseline_metrics = {
    "model": "DecisionTree",
    "tree_depth": int(dt.get_depth()),
    "n_leaves": int(dt.get_n_leaves()),
    "cv_f1_mean": round(float(cv_results["test_f1"].mean()), 4),
    "cv_f1_std": round(float(cv_results["test_f1"].std()), 4),
    "cv_precision_mean": round(float(cv_results["test_precision"].mean()), 4),
    "cv_recall_mean": round(float(cv_results["test_recall"].mean()), 4),
    "cv_accuracy_mean": round(float(cv_results["test_accuracy"].mean()), 4),
    "test_f1": round(float(test_f1), 4),
    "test_precision": round(float(test_prec), 4),
    "test_recall": round(float(test_rec), 4),
    "test_accuracy": round(float(test_acc), 4),
    "confusion_matrix": confusion_matrix(y_test, y_pred).tolist(),
    "feature_importances": {k: float(v) for k, v in importances.round(4).items()},
}

with open("baseline_metrics.json", "w") as f:
    json.dump(baseline_metrics, f, indent=2)

print("Baseline metrics saved to baseline_metrics.json ✓")
print(json.dumps(baseline_metrics, indent=2))